# 🎤 Kaan Ses Klonlama — Colab (Ücretsiz GPU)

`sesler/kaan.wav` örneğinden Kaan'ın sesini **Coqui XTTS-v2** ile klonlar; önce kısa bir test, sonra Kaan'ın klon sesiyle **tam bölüm** üretir.

**Önce:** üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin (XTTS CPU'da çok yavaştır).

> ⚠️ **Lisans:** XTTS-v2 (CPML) yalnızca **ticari olmayan** kullanım içindir. Videoları **para kazanma amaçlı** yayımlayacaksan Kaan için klonlama yerine `edge-tts` kullan.


## 1) GPU kontrolü


In [ ]:
!nvidia-smi -L || echo 'GPU YOK! Runtime > Change runtime type > T4 GPU secin.'

## 2) Depoyu klonla + coqui-tts kur
`sesler/kaan.wav` örnek sesi repoda hazır gelir — ayrıca yükleme gerekmez.
(Kendi ses örneğini kullanmak istersen sol paneldeki dosya gezgininden `sesler/kaan.wav` üzerine kendi 6–20 sn'lik temiz konuşma kaydını at.)


In [ ]:
%cd /content
![ -d 'angrav-ty' ] || git clone https://github.com/fahrimert99-cmd/angrav-ty.git
%cd /content/angrav-ty
!git pull -q || true
!pip -q install coqui-tts
import os; print('kaan.wav var mi:', os.path.exists('sesler/kaan.wav'))

## 3) Hızlı klon testi 🎧
Kaan'ın sesini klonlayıp kısa bir cümle söyletir. İlk çalıştırmada XTTS-v2 modeli iner (~2 GB, birkaç dk).


In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'   # lisans onayini non-interaktif kabul et
from TTS.api import TTS
import torch
cihaz = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(cihaz)
tts.tts_to_file(
    text='Merhaba! Ben Kaan, bu ailenin babasiyim. Bugun cok guzel bir gun, hadi oyun oynayalim!',
    speaker_wav='sesler/kaan.wav',
    language='tr',
    file_path='kaan_klon_test.wav')
from IPython.display import Audio, display
display(Audio('kaan_klon_test.wav'))

## 4) Tam bölüm üret — Kaan klon sesiyle 🎬
`config/karakterler.json` içinde `kaan` zaten `ses_motoru: "xtts"` + `ses: sesler/kaan.wav` ile tanımlı; diğer karakterler edge-tts'te kalır.
Senaryo için bir API anahtarı gir (Claude/Groq/Gemini). Hiçbiri yoksa anahtarsız `pollinations`'a düşer.


In [ ]:
import getpass, pathlib
# Sadece SENARYO icin saglayici sec: 'groq' | 'gemini' | 'claude' | 'pollinations'
saglayici = 'groq'
anahtar = getpass.getpass(f'{saglayici.upper()} API anahtari (pollinations icin bos birak): ').strip()
cfg = open('config/ayarlar.ornek.yaml', encoding='utf-8').read()
cfg = cfg.replace('saglayici: "gemini"', f'saglayici: "{saglayici}"')
pathlib.Path('config/ayarlar.yaml').write_text(cfg, encoding='utf-8')
# anahtari ortam degiskenine koy (senaryo.py oradan okur)
import os
env_ad = {'groq':'GROQ_API_KEY','gemini':'GEMINI_API_KEY','claude':'ANTHROPIC_API_KEY'}.get(saglayici)
if env_ad and anahtar: os.environ[env_ad] = anahtar
print('ayarlar hazir. Senaryo saglayici:', saglayici)

In [ ]:
!python main.py --konu "Kaan cocuklari Mira ve Ege ile bahcede saklambac oynuyor" --sure 45 --karakterler kaan,mira,ege

## 5) Sonucu izle / indir


In [ ]:
import glob, os
from IPython.display import Video, display
mp4 = sorted(glob.glob('cikti/**/*.mp4', recursive=True), key=os.path.getmtime)
assert mp4, 'Video bulunamadi — ustteki adimlari kontrol et.'
print('Video:', mp4[-1]); display(Video(mp4[-1], embed=True, width=640))
from google.colab import files; files.download(mp4[-1])

---
### Notlar
- Klon kalitesi örnek sesin kalitesine bağlıdır: **6–20 sn, tek kişi, gürültüsüz, net** konuşma en iyisi. `sesler/kaan.wav` şu an 16 kHz mono — daha yüksek kaliteli bir örnekle sonuç iyileşir.
- Kod tarafı hazır: `_xtts` modeli **bir kez** yükler (her replikte değil) ve **GPU** kullanır; `COQUI_TOS_AGREED=1` ile lisans onayı non-interaktif geçilir.
- XTTS başarısız olursa pipeline otomatik **edge-tts**'e düşer (üretim durmaz).
- **Lisans:** XTTS-v2 ticari değildir; monetize kanal için Kaan'ı da `edge-tts`'e alın (`config/karakterler.json` → kaan → `ses_motoru: "edge"`).
